# example / pyfunc / linear

**Intent**
- Takes `example_pyfunc_in`, a number.
- Outputs `example_pyfunc_out`, the input scaled by a fitted gain plus a fitted offset.
- Shows a fitted model end to end, with every number reproducible.
- Acceptance holds the rmse on `validation` and `backtest` to 0.5.
- It is a fixture, tested by `fab unittest` and `fab systest` and never released.

**Data**
- `training` is ten labelled points on a line with fixed noise, the only data the fit sees.
- `validation` is five more points, and `backtest` is ten further out, held back for acceptance.
- `golden` is six exactly representable inputs, then a request with the input missing that must give a signature error.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json
from os.path import join

import numpy as np

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "example/pyfunc/linear"
MODEL_NAME = "example_pyfunc_linear"
TARGET = {"kind": "embedded", "language": "python"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
PYTHON_DEPS = ["polars"]
TRAINING = "pyfunc_linear_training_2026-11-03_2026-11-03"
VALIDATION = "pyfunc_linear_validation_2026-11-03_2026-11-03"
BACKTEST = "pyfunc_linear_backtest_2026-11-03_2026-11-03"
GOLDEN = "pyfunc_linear_golden"
INPUT = INPUTS[0]
OUTPUT = OUTPUTS[0]

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {INPUT: xs, OUTPUT: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled(0, 10))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled(10, 5))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled(15, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
gain, offset = np.polyfit(training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
from workbench.model import evaluate_acceptance, measure
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
import polars as pl
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "example_pyfunc_in"
OUTPUT = "example_pyfunc_out"
GAIN = {gain!r}
OFFSET = {offset!r}


class Linear(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: (pl.Series(model_input[INPUT].to_list()) * GAIN + OFFSET).to_list()}})


set_model(Linear())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(gain=GAIN, offset=OFFSET))

predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        predicted = [predict({INPUT: [x]})[OUTPUT][0] for x in frame[INPUT].to_list()]
        checks.append({"split": split, "output": OUTPUT, "name": "rmse", "value": measure(frame[OUTPUT].to_list(), predicted)["rmse"], "limit": 0.5, "direction": "max"})
    return checks


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
def golden_samples():
    xs = [-1.0, 0.0, 0.25, 1.0, 2.5, 10.0]
    return {INPUT: [*xs, None], OUTPUT: [*[GAIN * x + OFFSET for x in xs], None], "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic line, training split",
        [training_path, golden_path],
        "least squares line, gain and offset built into model.py",
        implementation="mlflow.pyfunc model.py, requires mlflow, pandas and polars",
    ),
    model_path=model_path,
    python_deps=PYTHON_DEPS,
    evaluation=evaluation,
    sequence="across zero, fractions and a larger value",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped runtime")